# Full Video Eval (Colab)

Thin wrapper around `scripts/run_checkpoint_full_render_eval.py`.
It renders full-scene YUV outputs from the current checkpoint and then
computes metrics from the rendered NV12 videos.

## Required Google Drive files

Create this folder:

`MyDrive/ISP_colab/full_video_eval/`

Upload there:

- `full_video_eval_bundle.zip`
- `e2e_best.pth`
- source/reference files under `data/` only for the scenes listed in `SCENES`
  - possible scene pairs: `day_0.bin/.yuv`, `night_0.bin/.yuv`, `tunnel_0.bin/.yuv`
  - example: if `SCENES = 'night'`, upload only `night_0.bin` and `night_0.yuv`

Reports are copied to:

`MyDrive/ISP_colab/full_video_eval_outputs/`

Large rendered `*.yuv` videos are kept locally inside the Colab runtime
and are not copied back to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'full_video_eval'
DRIVE_OUTPUT = DRIVE_ROOT / 'full_video_eval_outputs'

CKPT_FILE = DRIVE_INPUT / 'e2e_best.pth'
DATA_DIR = DRIVE_INPUT / 'data'

LOCAL_REPO = Path('/content/ISP')
LOCAL_OUTPUT = Path('/content/full_video_eval_outputs')
LOCAL_CKPT = LOCAL_REPO / 'artifacts' / 'checkpoints' / 'e2e_quality' / 'e2e_best.pth'

MODE = 'isp_cnn'          # 'isp' or 'isp_cnn'
SCENES = 'day,night,tunnel'
METRIC_STRIDE = 1        # 1 = every frame
COMPUTE_IQA = True
FORCE_RENDER = False

print('Drive input :', DRIVE_INPUT)
print('Drive output:', DRIVE_OUTPUT)
print('Local output:', LOCAL_OUTPUT)
print('Local repo  :', LOCAL_REPO)

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Please switch Colab runtime to GPU before starting the full eval run.')

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

%pip -q install pyiqa

In [ ]:
import shutil
import time
import zipfile

def required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def required_one(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError('None of these paths exist:\n' + '\n'.join(str(p) for p in paths))

BUNDLE_ZIP = DRIVE_INPUT / 'full_video_eval_bundle.zip'
required(CKPT_FILE)
required(DATA_DIR)

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(BUNDLE_ZIP, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'Bundle unpacked in {time.time() - t0:.1f}s')

LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT_FILE, LOCAL_CKPT)

local_data_dir = LOCAL_REPO / 'data'
local_data_dir.mkdir(parents=True, exist_ok=True)
requested_scenes = [scene.strip() for scene in SCENES.split(',') if scene.strip()]
valid_scenes = ['day', 'night', 'tunnel']
unknown_scenes = [scene for scene in requested_scenes if scene not in valid_scenes]
if unknown_scenes:
    raise ValueError(f'Unknown scenes in SCENES: {unknown_scenes}')
present_files = sorted(p.name for p in DATA_DIR.iterdir() if p.is_file()) if DATA_DIR.exists() else []
print('Files found in data/:', present_files)

def find_input_file(name: str):
    direct = DATA_DIR / name
    if direct.exists():
        return direct
    matches = [
        p for p in DRIVE_INPUT.rglob(name)
        if p.is_file() and '.ipynb_checkpoints' not in p.parts
    ]
    if len(matches) == 1:
        print(f'Found {name} outside data/: {matches[0]}')
        return matches[0]
    if len(matches) > 1:
        raise FileNotFoundError(f'Multiple candidates found for {name}: {matches}')
    return None

scene_file_map = {}
available_scenes = []
for scene in valid_scenes:
    bin_path = find_input_file(f'{scene}_0.bin')
    yuv_path = find_input_file(f'{scene}_0.yuv')
    if bin_path and yuv_path:
        available_scenes.append(scene)
        scene_file_map[scene] = {'bin': bin_path, 'yuv': yuv_path}
    elif bin_path or yuv_path:
        raise FileNotFoundError(
            f'Incomplete files for scene {scene}: both .bin and .yuv are required'
        )
effective_scenes = [scene for scene in requested_scenes if scene in available_scenes]
missing_requested = [scene for scene in requested_scenes if scene not in available_scenes]
if missing_requested:
    raise FileNotFoundError(
        f'Missing requested scenes in {DATA_DIR}: {missing_requested}. '
        f'Available complete scenes: {available_scenes}. '
        f'Files found: {present_files}'
    )
if not effective_scenes:
    raise FileNotFoundError(f'No complete scene pairs found in {DATA_DIR}')
EFFECTIVE_SCENES = ','.join(effective_scenes)
required_scene_files = []
for scene in effective_scenes:
    required_scene_files.extend([f'{scene}_0.bin', f'{scene}_0.yuv'])
print('Selected scenes:', ', '.join(effective_scenes))

def copy_with_retries(src: Path, dst: Path, retries: int = 3, delay_s: float = 2.0):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as exc:
            last_exc = exc
            print(f'copy failed for {src.name} (attempt {attempt}/{retries}): {exc}')
            if attempt < retries:
                time.sleep(delay_s)
    raise last_exc

for name in required_scene_files:
    scene = name.split('_0.')[0]
    kind = 'bin' if name.endswith('.bin') else 'yuv'
    src = required(scene_file_map[scene][kind])
    dst = local_data_dir / name
    copy_with_retries(src, dst)
    print('copied', name, f'({dst.stat().st_size / 1024**3:.2f} GB)')

checks = [
    LOCAL_REPO / 'scripts' / 'run_checkpoint_full_render_eval.py',
    LOCAL_REPO / 'artifacts' / 'checkpoints' / 'optuna_tuning' / 'optuna_best_params.json',
    LOCAL_CKPT,
    LOCAL_REPO / 'data' / 'imx623.toml',
]
for path in checks:
    print(path.relative_to(LOCAL_REPO), 'OK' if path.exists() else 'MISSING')
    if not path.exists():
        raise FileNotFoundError(path)

eval_utils_text = (LOCAL_REPO / 'isp' / 'evaluation' / 'evaluation_utils.py').read_text(encoding='utf-8')
if 'even=False' not in eval_utils_text:
    raise RuntimeError('Extracted bundle still contains old VIF mode; expected even=False')
print('Bundle VIF mode: even=False')
print('Rendered YUV output dir (local):', LOCAL_OUTPUT)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-B', 'scripts/run_checkpoint_full_render_eval.py',
    '--device', 'cuda',
    '--ckpt', str(LOCAL_CKPT.relative_to(LOCAL_REPO)),
    '--optuna-best-json', 'artifacts/checkpoints/optuna_tuning/optuna_best_params.json',
    '--config', 'data/imx623.toml',
    '--norm-weights', 'artifacts/baselines/norm_weights.json',
    '--mode', MODE,
    '--scenes', EFFECTIVE_SCENES,
    '--metric-stride', str(METRIC_STRIDE),
    '--output-dir', str(LOCAL_OUTPUT),
]
if not COMPUTE_IQA:
    cmd += ['--no-iqa']
if FORCE_RENDER:
    cmd += ['--force-render']

print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd, cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'Full video eval script exited with code {ret}')

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
for path in LOCAL_OUTPUT.iterdir():
    if path.is_file():
        dst = DRIVE_OUTPUT / path.name
        shutil.copy2(path, dst)
        print('synced report:', dst)

local_videos = LOCAL_OUTPUT / 'videos'
if local_videos.exists():
    print('Rendered YUV files kept local in:', local_videos)

In [ ]:
import json

report_path = DRIVE_OUTPUT / f'report_{MODE}.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps(report['overall_metrics'], indent=2))
else:
    print('Missing report:', report_path)